<a href="https://colab.research.google.com/github/pravallikachejerla/Data-science/blob/main/saleprice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Load the datasets
train_df = pd.read_csv('/content/train.csv')
test_df = pd.read_csv('/content/test.csv')
sample_submission = pd.read_csv('/content/sample_submission.csv')

# Target variable
y = train_df['SalePrice']
train_df.drop(['SalePrice'], axis=1, inplace=True)

# Combine train and test data for preprocessing
all_data = pd.concat([train_df, test_df], keys=['train', 'test'])

# Handling missing values
# Fill numerical columns with the median value
num_cols = all_data.select_dtypes(include=[np.number]).columns
imputer = SimpleImputer(strategy='median')
all_data[num_cols] = imputer.fit_transform(all_data[num_cols])

# Fill categorical columns with the most frequent value
cat_cols = all_data.select_dtypes(include=[object]).columns
imputer = SimpleImputer(strategy='most_frequent')
all_data[cat_cols] = imputer.fit_transform(all_data[cat_cols])

# Encode categorical variables
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    all_data[col] = le.fit_transform(all_data[col])
    label_encoders[col] = le

# Split back into train and test data
train_data = all_data.loc['train'].drop('Id', axis=1)
test_data = all_data.loc['test'].drop('Id', axis=1)

# Feature scaling
scaler = StandardScaler()
train_data = scaler.fit_transform(train_data)
test_data = scaler.transform(test_data)

# Train-test split for validation
X_train, X_val, y_train, y_val = train_test_split(train_data, y, test_size=0.2, random_state=42)

# Train the model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predict and evaluate on validation set
val_preds = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(np.log1p(y_val), np.log1p(val_preds)))
print(f'Validation RMSE: {rmse}')

# Predict on the test set
test_preds = model.predict(test_data)

# Prepare the submission file
submission = pd.DataFrame({
    'Id': test_df['Id'],
    'SalePrice': test_preds
})

submission.to_csv('submission.csv', index=False)


Validation RMSE: 0.15309792726327923


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer

# Load the datasets
train_df = pd.read_csv('/content/train.csv')
test_df = pd.read_csv('/content/test.csv')
sample_submission = pd.read_csv('/content/sample_submission.csv')

# Target variable
y = train_df['SalePrice']

# Select features
features = ['GrLivArea', 'BedroomAbvGr', 'FullBath', 'HalfBath']
X = train_df[features]
X_test = test_df[features]

# Handling missing values
imputer = SimpleImputer(strategy='median')
X = imputer.fit_transform(X)
X_test = imputer.transform(X_test)

# Train the model
model = LinearRegression()
model.fit(X, y)

# Predict on the test set
test_preds = model.predict(X_test)

# Prepare the submission file
submission = pd.DataFrame({
    'Id': test_df['Id'],
    'SalePrice': test_preds
})

submission.to_csv('submission.csv', index=False)

print("Submission file created successfully.")


Submission file created successfully.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer

# Load the dataset
train_df = pd.read_csv('/content/train.csv')

# Target variable
y = train_df['SalePrice']

# Select features
features = ['GrLivArea', 'BedroomAbvGr', 'FullBath', 'HalfBath']
X = train_df[features]

# Handling missing values
imputer = SimpleImputer(strategy='median')
X = imputer.fit_transform(X)

# Train-test split for validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict and evaluate on validation set
val_preds = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(np.log1p(y_val), np.log1p(val_preds)))
print(f'Validation RMSE: {rmse}')


Validation RMSE: 0.27283096292144704


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

# Load the dataset
train_df = pd.read_csv('/content/train.csv')

# Target variable
y = train_df['SalePrice']

# Select more features
features = ['GrLivArea', 'BedroomAbvGr', 'FullBath', 'HalfBath', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'OverallQual', 'YearBuilt']
X = train_df[features]

# Handling missing values
imputer = SimpleImputer(strategy='median')
X = imputer.fit_transform(X)

# Train-test split for validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the linear regression model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predict and evaluate on validation set for Linear Regression
lr_val_preds = lr_model.predict(X_val)
lr_rmse = np.sqrt(mean_squared_error(np.log1p(y_val), np.log1p(lr_val_preds)))
print(f'Linear Regression Validation RMSE: {lr_rmse}')

# Train the Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predict and evaluate on validation set for Random Forest
rf_val_preds = rf_model.predict(X_val)
rf_rmse = np.sqrt(mean_squared_error(np.log1p(y_val), np.log1p(rf_val_preds)))
print(f'Random Forest Validation RMSE: {rf_rmse}')

# Cross-validation for Random Forest
cv_scores = cross_val_score(rf_model, X, y, cv=5, scoring='neg_mean_squared_error')
cv_rmse = np.sqrt(-cv_scores)
print(f'Random Forest Cross-Validation RMSE: {cv_rmse.mean()}')


Linear Regression Validation RMSE: 0.19564531787094308
Random Forest Validation RMSE: 0.17118238557991125
Random Forest Cross-Validation RMSE: 32168.767839635344


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.impute import SimpleImputer

# Load the dataset
train_df = pd.read_csv('/content/train.csv')

# Target variable
y = train_df['SalePrice']

# Select more features
features = ['GrLivArea', 'BedroomAbvGr', 'FullBath', 'HalfBath', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'OverallQual', 'YearBuilt']
X = train_df[features]

# Handling missing values
imputer = SimpleImputer(strategy='median')
X = imputer.fit_transform(X)

# Train-test split for validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the linear regression model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predict and evaluate on validation set for Linear Regression
lr_val_preds = lr_model.predict(X_val)
lr_rmse = np.sqrt(mean_squared_error(np.log1p(y_val), np.log1p(lr_val_preds)))
lr_r2 = r2_score(y_val, lr_val_preds)
print(f'Linear Regression Validation RMSE: {lr_rmse}')
print(f'Linear Regression Validation R²: {lr_r2}')

# Train the Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predict and evaluate on validation set for Random Forest
rf_val_preds = rf_model.predict(X_val)
rf_rmse = np.sqrt(mean_squared_error(np.log1p(y_val), np.log1p(rf_val_preds)))
rf_r2 = r2_score(y_val, rf_val_preds)
print(f'Random Forest Validation RMSE: {rf_rmse}')
print(f'Random Forest Validation R²: {rf_r2}')

# Cross-validation for Random Forest
cv_scores = cross_val_score(rf_model, X, y, cv=5, scoring='neg_mean_squared_error')
cv_rmse = np.sqrt(-cv_scores)
print(f'Random Forest Cross-Validation RMSE: {cv_rmse.mean()}')


Linear Regression Validation RMSE: 0.19564531787094308
Linear Regression Validation R²: 0.8000742140241333
Random Forest Validation RMSE: 0.17118238557991125
Random Forest Validation R²: 0.8866266453587799
Random Forest Cross-Validation RMSE: 32168.767839635344
